In [1]:
# in terminal: uv sync
import pandas as pd
import numpy as np
import re

# looading accounts dataset

accounts = pd.read_csv('../../data/raw/HI-Small_accounts.csv')

accounts.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [2]:
print(accounts["Bank Name"].unique())

<StringArray>
[        'Portugal Bank #4507',             'Canada Bank #27',
                 'UK Bank #33',          'Germany Bank #4815',
 'National Bank of Harrisburg',             'Spain Bank #439',
       'Savings Bank of Omaha',             'Brazil Bank #39',
             'Mexico Bank #16',             'Russia Bank #39',
 ...
      'Switzerland Bank #1397',               'UK Bank #1117',
           'Israel Bank #1124',            'Crytpo Bank #904',
             'Japan Bank #681',            'Brazil Bank #218',
           'Greece Bank #2917',            'Brazil Bank #411',
             'Japan Bank #701',                'UK Bank #483']
Length: 20053, dtype: str


## Bank locations

With this section the goal was to use python and regex to work on the bank names in order to extrapolate their locations. <br> Unfortunately, due to this being syntetic data I couldn't get very far with a code based approach and had to get creative and manually modify some of the features and research approaches to cathegorize the banks.

In [ ]:
# creating a new dataframe for mapping banks
bank_map = accounts[['Bank Name']].drop_duplicates().reset_index(drop=True)

In [4]:
def find_country(i):
    if re.search(r'Bank #\d+$', i):
        match = re.search(r'^([A-Za-z ]+?) Bank', i)
        if match:
            return match.group(1) 
    
    if re.search(r'Bank of', i):
        match = re.search(r'Bank of ([A-Za-z ]+?)$', i)
        if match:
            return match.group(1) 

    return 'other'

In [5]:
bank_map['Country'] = bank_map['Bank Name'].apply(find_country)

In [18]:
usa = [ "Albany", "Arkadelphia", "Augusta", "Billings", "Boston", "Bridgeport", "Butte", "Chicago",  "Cleveland", "Columbus", "Dallas",
        "Danbury", "Denver", "Detroit", "Fairfield", "Fort Wayne", "Harrisburg", "Hartford", "Helena", "Houston", "Huron", "Indianapolis",
        "Lacrosse", "Los Angeles", "Laramie", "Madison", "Miami", "Milford", "New Orleans", "New York", "Newport", "Omaha", "Philadelphia", "Phoenix",
        "Pittsburgh", "Plattsburg", "Portland", "Providence", "Sacramento", "Seattle", "Springfield", "Tampa", "the East", "the North",
        "the South", "the Valley", "the West", "Topeka", "Tuscon", "Watertown"
        ]
uk = ["Lincoln", "Newbury", "Portsmouth"]
can = ["Montpelier"]

In [13]:
bank_map['Country'] = bank_map['Country'].replace(usa, 'USA')
bank_map['Country'] = bank_map['Country'].replace(uk, 'UK')
bank_map['Country'] = bank_map['Country'].replace(can, 'Canada')

bank_map['Country'] = bank_map['Country'].replace('Crytpo', 'Crypto')

## Bank types

In [ ]:
def find_type(name):
    n = name.lower()
    
    if 'credit union' in n or 'cooperative' in n:
        return 'Cooperative'
    elif 'bancorp' in n:
        return 'Holding'
    elif 'trust' in n:
        return 'Trust'
    elif 'savings' in n or 'thrift' in n:
        return 'Savings'
    elif 'bank' in n:
        return 'Commercial'
    else:
        return None

In [16]:
bank_map['Type'] = bank_map['Bank Name'].apply(find_type)

In [20]:
# creating the mapping csv file
bank_map.to_csv("../../data/raw/bank_map.csv", index=False)